# Wikipedia monthly earthquakes: fill the gaps by HTML scraping (2000-2025)

The yearly pages `List_of_earthquakes_in_<year>` present each month in one of two ways: a **table**, or a **bulleted text list** (e.g. 2006, 2008, 2013-2016, and some months of 2007, 2010, 2012). `pandas.read_html` only sees the tables, so the text months were never saved.

This notebook goes through every year and every month and, for each expected file `earthquakes_<year>/<Month>_<year>.csv`:

1. **file exists with rows** -> skipped
2. **file missing, or exists but has no rows** (like `February_2007.csv`, header only) -> scraped from the saved HTML: first from a month table, otherwise from the bullet sentences

Scraped files use exactly the columns of the reference file `February_2007.csv`: `year, Date, Country and location, Mw, Depth (km), MMI, Notes, Casualties Dead, Casualties Injured`. No collection dates appear in any file name.

**Text months:** Wikipedia's bullets give no MMI and no casualty columns, so `MMI` is `-`, the full sentence remainder goes into `Notes`, and `Casualties Dead/Injured` are filled by a best-effort pattern match on that sentence (`-` when nothing is found). Every text-derived row that mentions deaths or injuries is written to `reports/wikipedia_html_scraping_review.csv` so the numbers can be checked by hand.

**Cross-check:** each month also has a small infobox with "Number by magnitude". The notebook compares its count of M6+ events with the M6+ rows in the final CSV and saves the comparison.

## 1. Imports, folders and the reference format

In [1]:
from pathlib import Path
from io import StringIO
from urllib.robotparser import RobotFileParser
from collections import defaultdict
import re
import time

import requests
import pandas as pd
from bs4 import BeautifulSoup

# Run this notebook from the EarthQuake project root.
PROJECT_DIR = Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
REPORT_DIR = PROJECT_DIR / "reports"

RAW_WIKI_DIR = RAW_DIR / "wikipedia_earthquake_list"
WIKI_INTERIM_DIR = INTERIM_DIR / "wikipedia_earthquake_list"

for folder in [RAW_WIKI_DIR, WIKI_INTERIM_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

START_YEAR, END_YEAR = 2000, 2025
PAGE_URL = "https://en.wikipedia.org/wiki/List_of_earthquakes_in_{year}"
ROBOTS_URL = "https://en.wikipedia.org/robots.txt"
USER_AGENT = "EarthquakeAcademicProject/1.0 (student research)"
REQUEST_DELAY_SECONDS = 1.0
MAX_ATTEMPTS = 3
MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

# Output format = the reference CSV (falls back to the same columns if the file is not there)
DEFAULT_COLUMNS = ["year", "Date", "Country and location", "Mw", "Depth (km)",
                   "MMI", "Notes", "Casualties Dead", "Casualties Injured"]
REFERENCE_CSV = WIKI_INTERIM_DIR / "earthquakes_2007" / "February_2007.csv"
if REFERENCE_CSV.exists():
    OUTPUT_COLUMNS = pd.read_csv(REFERENCE_CSV, nrows=0, encoding="utf-8-sig").columns.tolist()
else:
    OUTPUT_COLUMNS = DEFAULT_COLUMNS

print("Project:", PROJECT_DIR)
print("Interim folder:", WIKI_INTERIM_DIR)
print("Output columns:", OUTPUT_COLUMNS)

Project: d:\DA2402\Project\EarthQuake
Interim folder: d:\DA2402\Project\EarthQuake\data\interim\wikipedia_earthquake_list
Output columns: ['year', 'Date', 'Country and location', 'Mw', 'Depth (km)', 'MMI', 'Notes', 'Casualties Dead', 'Casualties Injured']


## 2. Check Wikipedia robots.txt

In [2]:
robots_response = session.get(ROBOTS_URL, timeout=(15, 60))
robots_response.raise_for_status()

(REPORT_DIR / "robots_wikipedia.txt").write_text(
    f"Requested URL: {ROBOTS_URL}\nHTTP status: {robots_response.status_code}\n\n" + robots_response.text,
    encoding="utf-8")

robots_parser = RobotFileParser()
robots_parser.parse(robots_response.text.splitlines())

sample_url = PAGE_URL.format(year=2007)
robots_allowed = robots_parser.can_fetch(USER_AGENT, sample_url)
print("Example page:", sample_url, "| allowed by robots rules:", robots_allowed)
print("Terms of Use: https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use")
print("User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy")
if not robots_allowed:
    raise RuntimeError("Wikipedia robots.txt does not allow the target page for this user agent.")

Example page: https://en.wikipedia.org/wiki/List_of_earthquakes_in_2007 | allowed by robots rules: True
Terms of Use: https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use
User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy


## 3. Raw HTML: reuse the saved pages, download only what is missing

Pages already in `data/raw/wikipedia_earthquake_list/` are reused without a new request.

In [3]:
def get_year_html(year):
    """Return the page HTML for a year (cached file first, otherwise download and save unchanged)."""
    url = PAGE_URL.format(year=year)
    raw_path = RAW_WIKI_DIR / f"List_of_earthquakes_in_{year}.html"

    if raw_path.exists() and raw_path.stat().st_size > 0:
        return raw_path.read_text(encoding="utf-8", errors="replace"), "reused cached HTML"

    if not robots_parser.can_fetch(USER_AGENT, url):
        return None, "disallowed by robots.txt"

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = session.get(url, timeout=(15, 60))
            if response.status_code == 404:
                return None, "404"
            response.raise_for_status()
            raw_path.write_bytes(response.content)          # unchanged response bytes
            time.sleep(REQUEST_DELAY_SECONDS)
            return response.text, "downloaded"
        except requests.RequestException as exc:
            print(f"{year}: attempt {attempt}/{MAX_ATTEMPTS} failed: {exc}")
            time.sleep(2 * attempt)
    return None, "failed after retries"


page_html, page_status = {}, {}
for year in range(START_YEAR, END_YEAR + 1):
    page_html[year], page_status[year] = get_year_html(year)
    print(year, page_status[year])

pd.DataFrame({"year": list(page_status), "status": list(page_status.values())}).to_csv(
    REPORT_DIR / "wikipedia_html_scraping_request_log.csv", index=False)

2000 reused cached HTML
2001 reused cached HTML
2002 reused cached HTML
2003 reused cached HTML
2004 reused cached HTML
2005 reused cached HTML
2006 reused cached HTML
2007 reused cached HTML
2008 reused cached HTML
2009 reused cached HTML
2010 reused cached HTML
2011 reused cached HTML
2012 reused cached HTML
2013 reused cached HTML
2014 reused cached HTML
2015 reused cached HTML
2016 reused cached HTML
2017 reused cached HTML
2018 reused cached HTML
2019 reused cached HTML
2020 reused cached HTML
2021 reused cached HTML
2022 reused cached HTML
2023 reused cached HTML
2024 reused cached HTML
2025 reused cached HTML


## 4. Parsing helpers

**How a month is found.** The page is read in document order. Each heading sets the "current section"; only a heading whose text is a month name starts a month. Everything that follows until the next heading belongs to that month: its `wikitable` tables, its infobox, and its `<li>` bullets.

**How a bullet becomes a row.** A sentence such as

> A magnitude 6.9 earthquake struck Noto, Japan on March 25 at a depth of 8.0 km (5.0 mi). One person was killed, 356 were injured, ...

is split into magnitude, location (kept as written), day, depth and the remaining text (Notes). Deaths and injuries are read from the Notes with several patterns (`killed 112 people`, `One person was killed`, `356 were injured`, `N deaths`, ...). Number words up to twenty and tens are understood.

Text-derived rows keep Wikipedia's text order for the location (place, region, **country last**), whereas the table months put the **country first**. Handle both conventions when extracting the country in the cleaning step.

In [4]:
# ---------- generic helpers ----------
def clean_text(tag):
    return re.sub(r"\s+", " ", tag.get_text()).strip()


def li_own_text(li):
    """Text of a bullet without its nested sub-lists."""
    clone = BeautifulSoup(str(li), "lxml").find("li")
    for sub in clone.find_all(["ul", "ol"]):
        sub.decompose()
    return clean_text(clone)


def month_of(heading_text):
    h = re.sub(r"\[.*?\]", "", heading_text).strip().lower()
    return next((m for m in MONTHS if m.lower() == h), None)


def flatten_columns(columns):
    """Flatten a pandas MultiIndex header without changing table values."""
    flattened = []
    for column in columns:
        parts = column if isinstance(column, tuple) else (column,)
        parts = [str(p).strip() for p in parts if str(p).strip() and not str(p).startswith("Unnamed")]
        flattened.append(" ".join(dict.fromkeys(parts)) or "column")
    seen, unique = defaultdict(int), []
    for name in flattened:
        seen[name] += 1
        unique.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return unique


# ---------- text (bullet) parsing ----------
NUM_WORDS = {w: i for i, w in enumerate(
    ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven",
     "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen", "twenty"])}
NUM_WORDS.update({"thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90})
NUM = r"(?P<n>\d[\d,]*|" + "|".join(k for k in NUM_WORDS if k != "zero") + r")"
_W = r"(?:[A-Za-z'-]+\s+){0,2}?"                    # up to two filler words ("people", "people were")

DEATH_PATTERNS = [
    re.compile(rf"\b(?:killed|killing|kills|death toll of|deaths? of|lives of)\s+{NUM}\b", re.I),
    re.compile(rf"\b{NUM}\s+{_W}(?:(?:were|was)\s+)?(?:killed|died|dead|perished)\b", re.I),
    re.compile(rf"\b{NUM}\s+(?:deaths|fatalities|fatality|death)\b", re.I),
    re.compile(r"\b(?:a|an)\s+(?:person|man|woman|child|boy|girl|tourist|worker|motorist)\s+(?:was\s+)?(?:killed|died)\b", re.I),
]
INJURY_PATTERNS = [
    re.compile(rf"\b(?:injured|injuring|injuries to|wounding)\s+{NUM}\b", re.I),
    re.compile(rf"\b{NUM}\s+{_W}(?:(?:were|was)\s+)?(?:injured|hurt|wounded)\b", re.I),
    re.compile(rf"\b{NUM}\s+injuries\b", re.I),
    re.compile(r"\b(?:a|an)\s+(?:person|man|woman|child|boy|girl|tourist|worker|motorist)\s+(?:was\s+)?(?:injured|hurt)\b", re.I),
]
DEATH_WORD = re.compile(r"kill|dead|death|died|fatal|perish|casualt", re.I)
INJURY_WORD = re.compile(r"injur|hurt|wound", re.I)


def to_int(token):
    token = token.lower().replace(",", "")
    return int(token) if token.isdigit() else NUM_WORDS.get(token)


def first_count(patterns, text):
    for p in patterns:
        m = p.search(text)
        if not m:
            continue
        if "n" not in p.groupindex:                  # "a person was killed" -> 1
            return 1
        value = to_int(m.group("n"))
        if value is not None:
            return value
    return None


MONTH_RE = "|".join(MONTHS)
EVENT_RE = re.compile(
    rf"magnitude\s+(?P<mag>\d+(?:\.\d+)?)\s+(?:earthquake|quake)s?\s+"
    rf"(?:struck|hit|shook|occurred(?:\s+(?:in|near|off|at))?)\s+(?P<loc>.+?)\s+on\s+"
    rf"(?:(?P<m1>{MONTH_RE})\s+(?P<d1>\d{{1,2}})|(?P<d2>\d{{1,2}})\s+(?P<m2>{MONTH_RE}))\b(?P<rest>.*)$",
    re.S)
DEPTH_RE = re.compile(r"depth\s+of\s+(?P<depth>\d+(?:\.\d+)?)\s*km(?:\s*\([^)]*\))?", re.I)


def parse_event_text(text):
    m = EVENT_RE.search(text)
    if not m:
        return None
    rest = m.group("rest")
    dm = DEPTH_RE.search(rest)
    notes = rest[dm.end():] if dm else rest
    notes = re.sub(r"^[\s,;.:–-]+", "", notes).strip()
    return {
        "day": str(int(m.group("d1") or m.group("d2"))),
        "event_month": m.group("m1") or m.group("m2"),
        "mag": m.group("mag"),
        "loc": m.group("loc").strip(),
        "depth": dm.group("depth") if dm else "-",
        "notes": notes,
        "sentence": text,
    }


def events_to_df(events, year):
    rows = []
    for ev in sorted(events, key=lambda e: int(e["day"])):
        dead = first_count(DEATH_PATTERNS, ev["notes"])
        injured = first_count(INJURY_PATTERNS, ev["notes"])
        rows.append({"year": year, "Date": ev["day"], "Country and location": ev["loc"], "Mw": ev["mag"],
                     "Depth (km)": ev["depth"], "MMI": "-", "Notes": ev["notes"] or "-",
                     "Casualties Dead": "-" if dead is None else str(dead),
                     "Casualties Injured": "-" if injured is None else str(injured)})
    return pd.DataFrame(rows).reindex(columns=OUTPUT_COLUMNS, fill_value="-")


# ---------- table and infobox parsing ----------
def table_to_df(table, year):
    df = pd.read_html(StringIO(str(table)), keep_default_na=False, thousands=None)[0]
    df.columns = flatten_columns(df.columns)
    if not any(str(c).strip().lower() == "year" for c in df.columns):
        df.insert(0, "year", year)
    return df


def is_month_table(df):
    cols = [str(c).lower() for c in df.columns]
    return "date" in cols and any("location" in c for c in cols)


def parse_infobox(table):
    """'Number by magnitude' rows -> {lower bound: count}, e.g. {8: 1, 7: 0, 6: 13}."""
    counts = {}
    for tr in table.find_all("tr"):
        th, td = tr.find("th"), tr.find("td")
        if th is None or td is None:
            continue
        m = re.match(r"^(\d)\.\d\s*(?:[–\-]\s*\d\.\d|\+)?$", clean_text(th))
        digits = re.sub(r"\D", "", clean_text(td))
        if m and digits:
            counts[int(m.group(1))] = int(digits)
    return counts or None


# ---------- one pass over a yearly page ----------
def parse_year_html(html):
    soup = BeautifulSoup(html, "lxml")
    for tag in soup.select(".mw-editsection, sup.reference, span.sortkey, style"):
        tag.decompose()
    root = soup.select_one("div.mw-parser-output") or soup

    out = {m: {"tables": [], "events": [], "infobox": None} for m in MONTHS}
    current = None
    for el in root.find_all(["h2", "h3", "h4", "table", "li"]):
        if el.name in ("h2", "h3", "h4"):
            current = month_of(el.get_text())
            continue
        if current is None or el.find_parent("table"):
            continue
        if el.name == "table":
            classes = el.get("class") or []
            if "wikitable" in classes:
                out[current]["tables"].append(el)
            elif "infobox" in classes and out[current]["infobox"] is None:
                out[current]["infobox"] = parse_infobox(el)
        else:                                                # li
            text = li_own_text(el) if el.find(["ul", "ol"]) else clean_text(el)
            event = parse_event_text(text)
            if event:
                out[current]["events"].append(event)
    return out

## 5. Check each month's file, skip or scrape

Status values in the coverage report:

- `exists - skipped`: the CSV already has rows
- `scraped from table` / `scraped from text`: a missing or empty file was filled from the HTML
- `no events found in HTML`: nothing could be read for that month (check the page by hand)
- `no HTML for year`: the yearly page was not available

In [5]:
def existing_month_files(year_dir, month, year):
    return sorted(year_dir.glob(f"{month}*_{year}.csv"))


def count_rows(path):
    try:
        return len(pd.read_csv(path, keep_default_na=False, encoding="utf-8-sig"))
    except (pd.errors.EmptyDataError, ValueError):
        return 0


def count_mw6_plus(files):
    total = 0
    for path in files:
        try:
            df = pd.read_csv(path, keep_default_na=False, encoding="utf-8-sig")
        except (pd.errors.EmptyDataError, ValueError):
            continue
        col = next((c for c in df.columns if str(c).lower() in ("mw", "magnitude")), None)
        if col is not None:
            total += int(pd.to_numeric(df[col], errors="coerce").ge(6).sum())
    return total


def write_csv(df, path):
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
    except PermissionError as exc:
        raise PermissionError(f"Cannot write {path}. Close it in Excel and rerun.") from exc


def process_year(year, html, year_dir):
    coverage, review, crosscheck = [], [], []
    parsed = parse_year_html(html) if html else None

    for month in MONTHS:
        files = existing_month_files(year_dir, month, year)
        n_existing = sum(count_rows(p) for p in files)
        prior = "file missing" if not files else "file empty"
        info = parsed[month] if parsed else None

        if n_existing > 0:
            status, source, n_rows = "exists - skipped", "existing csv", n_existing

        elif parsed is None:
            status, source, n_rows = "no HTML for year", "none", 0

        else:
            tables = []
            for t in info["tables"]:
                try:
                    df = table_to_df(t, year)
                except ValueError:                   # header-only or malformed table
                    continue
                if is_month_table(df) and len(df) > 0:
                    tables.append(df)

            if tables:
                for i, df in enumerate(tables, start=1):
                    name = f"{month}_{year}.csv" if i == 1 else f"{month}_part{i}_{year}.csv"
                    write_csv(df, year_dir / name)
                status, source, n_rows = "scraped from table", "html table", sum(len(d) for d in tables)

            elif info["events"]:
                df = events_to_df(info["events"], year)
                write_csv(df, year_dir / f"{month}_{year}.csv")
                status, source, n_rows = "scraped from text", "html text bullets", len(df)

                for ev, (_, row) in zip(sorted(info["events"], key=lambda e: int(e["day"])), df.iterrows()):
                    dead_n = row["Casualties Dead"] != "-"
                    inj_n = row["Casualties Injured"] != "-"
                    needs = (bool(DEATH_WORD.search(ev["notes"])) and not dead_n) or \
                            (bool(INJURY_WORD.search(ev["notes"])) and not inj_n)
                    mismatch = ev["event_month"] != month
                    if DEATH_WORD.search(ev["notes"]) or INJURY_WORD.search(ev["notes"]) or mismatch:
                        review.append({"year": year, "month": month, "Date": ev["day"],
                                       "Country and location": ev["loc"], "Mw": ev["mag"],
                                       "deaths_extracted": row["Casualties Dead"],
                                       "injured_extracted": row["Casualties Injured"],
                                       "needs_review": needs or mismatch,
                                       "date_month_mismatch": mismatch,
                                       "sentence": ev["sentence"]})
            else:
                status, source, n_rows = "no events found in HTML", "none", 0

            if n_existing == 0 and status.startswith("scraped"):
                status = f"{status} ({prior})"

        coverage.append({"year": year, "month": month, "status": status, "source": source, "rows": n_rows})

        # cross-check against the month infobox ("Number by magnitude")
        final_files = existing_month_files(year_dir, month, year)
        box = info["infobox"] if info else None
        box_mw6 = sum(v for k, v in box.items() if k >= 6) if box else None
        csv_mw6 = count_mw6_plus(final_files)
        crosscheck.append({"year": year, "month": month, "csv_rows": sum(count_rows(p) for p in final_files),
                           "csv_mw6_plus": csv_mw6, "infobox_mw6_plus": box_mw6,
                           "difference": None if box_mw6 is None else csv_mw6 - box_mw6})
    return coverage, review, crosscheck


coverage_all, review_all, crosscheck_all = [], [], []

for year in range(START_YEAR, END_YEAR + 1):
    year_dir = WIKI_INTERIM_DIR / f"earthquakes_{year}"
    year_dir.mkdir(parents=True, exist_ok=True)
    cov, rev, xc = process_year(year, page_html[year], year_dir)
    coverage_all += cov; review_all += rev; crosscheck_all += xc

    counts = pd.Series([c["status"].split(" (")[0] for c in cov]).value_counts().to_dict()
    print(year, counts)

2000 {'exists - skipped': 12}
2001 {'exists - skipped': 12}
2002 {'exists - skipped': 12}
2003 {'exists - skipped': 12}
2004 {'exists - skipped': 12}
2005 {'exists - skipped': 12}
2006 {'scraped from text': 12}
2007 {'exists - skipped': 6, 'scraped from text': 6}
2008 {'scraped from text': 9, 'no events found in HTML': 3}
2009 {'exists - skipped': 12}
2010 {'scraped from text': 7, 'exists - skipped': 5}
2011 {'exists - skipped': 12}
2012 {'scraped from text': 10, 'exists - skipped': 2}
2013 {'scraped from text': 12}
2014 {'scraped from text': 12}
2015 {'scraped from text': 12}
2016 {'scraped from text': 12}
2017 {'exists - skipped': 12}
2018 {'exists - skipped': 12}
2019 {'exists - skipped': 12}
2020 {'exists - skipped': 12}
2021 {'exists - skipped': 12}
2022 {'exists - skipped': 12}
2023 {'exists - skipped': 12}
2024 {'exists - skipped': 12}
2025 {'exists - skipped': 12}


## 6. Coverage after the run and cross-check results

In [6]:
coverage_df = pd.DataFrame(coverage_all)
review_df = pd.DataFrame(review_all)
crosscheck_df = pd.DataFrame(crosscheck_all)

coverage_df.to_csv(REPORT_DIR / "wikipedia_month_coverage.csv", index=False)
review_df.to_csv(REPORT_DIR / "wikipedia_html_scraping_review.csv", index=False, encoding="utf-8-sig")
crosscheck_df.to_csv(REPORT_DIR / "wikipedia_infobox_crosscheck.csv", index=False)

print("Status counts over all 312 year-months:")
display(coverage_df["status"].str.split(" \(").str[0].value_counts().to_frame("year_months"))

# year x month grid: E = existing csv, T = scraped from table, X = scraped from text, . = nothing found
code_map = {"existing csv": "E", "html table": "T", "html text bullets": "X", "none": "."}
grid = (coverage_df.assign(code=coverage_df["source"].map(code_map))
        .pivot(index="year", columns="month", values="code")[MONTHS])
display(grid)

problems = coverage_df[coverage_df["source"] == "none"]
print("Months with nothing found (check these pages by hand):", len(problems))
display(problems)

print("\nCross-check vs infobox 'Number by magnitude' (months where an infobox was found):")
checked = crosscheck_df.dropna(subset=["infobox_mw6_plus"])
print("Months with an infobox:", len(checked), "| exact matches:", int((checked["difference"] == 0).sum()))
display(checked[checked["difference"] != 0])

print("\nText-derived rows that need a manual look (casualty words but no number, or date/month mismatch):")
if len(review_df):
    display(review_df[review_df["needs_review"]])

Status counts over all 312 year-months:


,year_months
status,
exists - skipped,217
scraped from text,92
no events found in HTML,3


month,January,February,March,April,May,June,July,August,September,October,November,December
year,,,,,,,,,,,,
2000,E,E,E,E,E,E,E,E,E,E,E,E
2001,E,E,E,E,E,E,E,E,E,E,E,E
2002,E,E,E,E,E,E,E,E,E,E,E,E
2003,E,E,E,E,E,E,E,E,E,E,E,E
2004,E,E,E,E,E,E,E,E,E,E,E,E
2005,E,E,E,E,E,E,E,E,E,E,E,E
2006,X,X,X,X,X,X,X,X,X,X,X,X
2007,E,X,X,X,X,X,X,E,E,E,E,E
2008,X,X,X,X,X,X,X,X,X,.,.,.


Months with nothing found (check these pages by hand): 3


,year,month,status,source,rows
105,2008,October,no events found in HTML,none,0
106,2008,November,no events found in HTML,none,0
107,2008,December,no events found in HTML,none,0



Cross-check vs infobox 'Number by magnitude' (months where an infobox was found):
Months with an infobox: 310 | exact matches: 227


,year,month,csv_rows,csv_mw6_plus,infobox_mw6_plus,difference
5,2000,June,28,25,26.0,-1.0
25,2002,February,10,6,4.0,2.0
61,2005,February,18,15,16.0,-1.0
74,2006,March,14,10,11.0,-1.0
75,2006,April,13,12,18.0,-6.0
...,...,...,...,...,...,...
262,2021,November,21,16,18.0,-2.0
263,2021,December,32,13,14.0,-1.0
268,2022,May,22,12,13.0,-1.0
273,2022,October,17,7,6.0,1.0



Text-derived rows that need a manual look (casualty words but no number, or date/month mismatch):


,year,month,Date,Country and location,Mw,deaths_extracted,injured_extracted,needs_review,date_month_mismatch,sentence
0,2006,January,8,"Peloponnese Peninsula, Greece,",6.7,-,-,True,False,A magnitude 6.7 earthquake struck Peloponnese ...
19,2006,June,13,"Gjirokaster County, Albania",4.5,-,-,True,False,A magnitude 4.5 earthquake struck Gjirokaster ...
22,2006,July,17,"south of Java, Indonesia",7.7,-,-,True,False,A magnitude 7.7 earthquake struck south of Jav...
28,2006,October,15,Hawaii,6.7,-,-,True,False,A magnitude 6.7 earthquake struck Hawaii on Oc...
78,2010,November,3,"Kraljevo, Serbia",5.3,2,-,True,False,"A magnitude 5.3 earthquake struck Kraljevo, Se..."
86,2012,April,11,"Indonesia, off northern Sumatra,",8.6,10,-,True,False,"A magnitude 8.6 earthquake hit Indonesia, off ..."
91,2012,May,22,24 km west of Bulgaria's capital Sofia,5.6,-,-,True,False,A magnitude 5.6 earthquake struck 24 km west o...
97,2012,August,26,"offshore El Salvador,",7.4,-,-,True,False,A magnitude 7.4 earthquake struck offshore El ...
98,2012,August,31,"offshore the Philippines, near Guiuan, Samar",7.6,1,-,True,False,A magnitude 7.6 earthquake struck offshore the...
99,2012,October,25,"Mormanno, Italy,",5.5,-,-,True,False,"A magnitude 5.5 earthquake struck Mormanno, It..."


## Notes for the cleaning stage

- Every month file now exists in the same 9-column layout. The coverage report says which source each month came from (`existing csv`, `html table`, `html text bullets`); use it to pick the right country convention (table: country first; text: country last).
- Text months have `MMI = -` and casualty counts that were pattern-matched from the sentence. The full sentence stays in `Notes`, and `reports/wikipedia_html_scraping_review.csv` lists every row with casualty words so the counts can be verified. Treat `-` in the casualty columns of text months as "not extracted", not as "zero deaths".
- Table months still contain repeated merged-cell values (e.g. the same death toll on several rows of one disaster), as in the original extraction.
- A difference against the infobox count is a prompt to look, not proof of an error: the infobox is hand-maintained and sometimes disagrees with the month's own list.
- Wikipedia text is CC BY-SA 4.0. Attribute the yearly pages in the report.